In [1]:
# Install required packages
%pip install ultralytics roboflow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.1 MB/s eta 0:00:00


In [2]:
# Verify GPU is available
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU device     : {torch.cuda.get_device_name(0)
                          if torch.cuda.is_available() else 'None'}")

CUDA available : True
GPU device     : Tesla T4


In [3]:
!git clone https://github.com/NaveenSa98/construction-safety-monitor.git
%cd construction-safety-monitor

Cloning into 'construction-safety-monitor'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 61 (delta 21), reused 51 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 20.04 KiB | 10.02 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/construction-safety-monitor


In [4]:
# Download base PPE dataset

from roboflow import Roboflow

rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(27)
dataset = version.download("yolov8", location="data/raw/base_dataset")



print("Base dataset downloaded successfully.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/base_dataset in yolov8:: 100%|██████████| 5610/5610 [00:00<00:00, 6593.64it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Base dataset downloaded successfully.


In [5]:
# Download coustom annotated dataset

from roboflow import Roboflow

# Source 1: Goggles PPE dataset
rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("tesis-n7wva").project("goggles-ppe")
version = project.version(2)
dataset = version.download("yolov8", location="data/raw/goggles_dataset" )

print("Goggles dataset downloaded.")

# Source 2: PPE dataset
rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("safety-jmser").project("safety_ppe")
version = project.version(23)
dataset = version.download("yolov8", location="data/raw/ppe_dataset")


print("Footwear dataset downloaded.")

# Source 3: Safety Gloves dataset
rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("roboflow-universe-projects").project("safety-gloves-xbnf8")
version = project.version(2)
dataset = version.download("yolov8", location="data/raw/gloves_dataset")

# Source 4: Footwear dataset
rf = Roboflow(api_key="VAK7OsUK4qpq5Xp9yMoS")
project = rf.workspace("ginting-inka-elgina-f6nat").project("safety-shoes")
version = project.version(1)
dataset = version.download("yolov8", location="data/raw/footwear_dataset")


print("\nAll custom datasets downloaded successfully.")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/goggles_dataset in yolov8:: 100%|██████████| 370/370 [00:00<00:00, 8987.41it/s]


Goggles dataset downloaded.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/ppe_dataset in yolov8:: 100%|██████████| 13270/13270 [00:04<00:00, 3030.27it/s]


Footwear dataset downloaded.
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/gloves_dataset in yolov8:: 100%|██████████| 6758/6758 [00:01<00:00, 6630.48it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/raw/footwear_dataset in yolov8:: 100%|██████████| 152/152 [00:00<00:00, 5104.68it/s]


All custom datasets downloaded successfully.


In [6]:
# Verify class lists for all datasets before running preparation
import yaml

datasets_to_check = {
    "Base Dataset"     : "data/raw/base_dataset/data.yaml",
    "Goggles Dataset"  : "data/raw/goggles_dataset/data.yaml",
    "Footwear Dataset" : "data/raw/footwear_dataset/data.yaml",
    "PPE Dataset"      : "data/raw/ppe_dataset/data.yaml",
    "Gloves Dataset"   : "data/raw/gloves_dataset/data.yaml",
}

for name, yaml_path in datasets_to_check.items():
    with open(yaml_path, "r") as f:
        content = yaml.safe_load(f)
    print(f"\n{name}:")
    for i, cls in enumerate(content.get("names", [])):
        print(f"  {i}: {cls}")


Base Dataset:
  0: Hardhat
  1: Mask
  2: NO-Hardhat
  3: NO-Mask
  4: NO-Safety Vest
  5: Person
  6: Safety Cone
  7: Safety Vest
  8: machinery
  9: vehicle

Goggles Dataset:
  0: Goggles
  1: NO-Goggles

Footwear Dataset:
  0: safety-shoes

PPE Dataset:
  0: No_BreathingApparatus
  1: No_Glove
  2: No_Goggles
  3: No_Harness
  4: No_Helmet
  5: No_Shoe
  6: Person

Gloves Dataset:
  0: Gloves
  1: NO-Gloves


In [7]:
# RUn the data preparation pipeline
!python src/data_preparation.py


# Verify the output structure
import os
for split in ["train", "val", "test"]:
    img_count = len(os.listdir(f"data/processed/images/{split}"))
    lbl_count = len(os.listdir(f"data/processed/labels/{split}"))
    print(f"{split:5s} — images: {img_count:4d} | labels: {lbl_count:4d}")


[INFO] Starting data preparation pipeline...
[INFO] Collecting pairs from base dataset...
[INFO] Base dataset pairs       : 2799
[INFO] Collecting pairs from goggles dataset...
[INFO] Goggles dataset pairs    : 179
[INFO] Collecting pairs from footwear dataset...
[INFO] Footwear dataset pairs   : 70
[INFO] Collecting pairs from ppe dataset...
[INFO] PPE dataset pairs   : 6629
[INFO] Collecting pairs from gloves dataset...
[INFO] Gloves dataset pairs     : 3373
[INFO] Total valid entries after remapping: 11572
[INFO] train split: 8100 images
[INFO] val   split: 2314 images
[INFO] test  split: 1158 images
[INFO] All 11572 image-label pairs written to /content/construction-safety-monitor/data/processed
[INFO] dataset.yaml written to: /content/construction-safety-monitor/data/processed/dataset.yaml
[INFO] Data preparation pipeline complete.
train — images: 8100 | labels: 8100
val   — images: 2314 | labels: 2314
test  — images: 1158 | labels: 1158


In [8]:
# Confirm dataset.yaml is correct before training
with open("data/processed/dataset.yaml", "r") as f:
    content = yaml.safe_load(f)

print("dataset.yaml contents:")
for key, value in content.items():
    print(f"  {key}: {value}")

dataset.yaml contents:
  path: /content/construction-safety-monitor/data/processed
  train: images/train
  val: images/val
  test: images/test
  nc: 12
  names: ['Person', 'Hardhat', 'NO-Hardhat', 'Safety Vest', 'NO-Safety Vest', 'Safety Gloves', 'NO-Safety Gloves', 'Safety Boots', 'NO-Safety Boots', 'Safety Goggles', 'NO-Safety Goggles', 'NO-Safety Harness']


In [ ]:
# Launch YOLOv8 fine-tuning
!python src/train.py

[INFO] Initialising YOLOv8 model...
[INFO] Pretrained weights loaded : yolov8m.pt
[INFO] Dataset                   : /content/construction-safety-monitor/data/processed/dataset.yaml
[INFO] Epochs                    : 50
[INFO] Image size                : 640
[INFO] Batch size                : 16
[INFO] Output directory          : /content/construction-safety-monitor/outputs/training_runs/ppe_detection_v1
[INFO] Starting training...

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/construction-safety-monitor/data/processed/dataset.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, er

In [ ]:
# Display training results
from IPython.display import Image, display
import glob
import os

results_dir = "outputs/training_runs/ppe_detection_v1"

print(f"Files in {results_dir}:")
for file in os.listdir(results_dir):
    print(f"- {file}")

# Show results plot (auto-generated by Ultralytics)
results_plot = f"{results_dir}/results.png"
display(Image(results_plot))

# Show confusion matrix
confusion_matrix = f"{results_dir}/confusion_matrix.png"
display(Image(confusion_matrix))

# Show F1 curve
f1_curve = f"{results_dir}/BoxF1_curve.png"
display(Image(f1_curve))